In [ ]:
import torch
print(torch.__version__)  # Check PyTorch version
print(torch.cuda.is_available())  # Should return True
print(torch.cuda.device_count())  # Should return the number of GPUs
print(torch.cuda.get_device_name(0))  # Should print the name of your GPU
from transformers import AutoTokenizer
from transformers import LlamaConfig
from transformers import AutoTokenizer
import os
from llama_model_query import LlamaModelQuery
from dataset_utils import get_c4
from process_cache import computing_cache,compute_proj_SVD_mem,compute_proj_SVD_Eigen_mem, compute_proj_KQT_mem
from proj_utils import read_proj,read_spectrums,computing_rank
from evaluation import validation_error

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device:",device)

In [ ]:
#Loading tokenizer
model_path = "path to model here"
tokenizer = AutoTokenizer.from_pretrained(model_path)#dirname(model_path))


model_config = LlamaConfig.from_pretrained(model_path)

In [ ]:
model = LlamaModelQuery.from_pretrained(model_path, device_map='auto')

In [4]:
#list of hyper parameters
D = model_config.hidden_size #full hidden dimension D=h*d
h = model_config.num_attention_heads #number of attention heads
hkv = model_config.num_key_value_heads #number of key/values heads, will be smaller than h if GQA
d = model_config.head_dim
L = model_config.num_hidden_layers

In [ ]:
#loading c4
traindata,valdata = get_c4(128,32,0,2048,model_path)

In [ ]:
#generating the cache
computing_cache(model,"Llama-2-7B_C4_train_128_2048",traindata,len(traindata))

In [ ]:
#computing projections SVD method
compute_proj_SVD_mem(model,"Llama-2-7B_C4_train_128_2048",128,"Llama-2-7B_projSVD_C4_train_128_2048",save=True)

In [ ]:
#computing projections Eigen
compute_proj_SVD_Eigen_mem(model_config,"Llama-2-7B_C4_train_128_2048",128,"Llama-2-7B_projSVDEigen_C4_train_128_2048",save=True)

In [ ]:
#experiments  with unbalance factor
for alpha in [2,3,4,5,6,7,8,9,10]:
    print(alpha)
    compute_proj_SVD_Eigen_mem(model_config,"Llama-2-7B_C4_train_128_2048",128,"Llama-2-7B_projSVDEigen_C4_train_128_2048_"+str(alpha),alpha=alpha,save=True)

In [ ]:
#computing projections KQ-SVD
compute_proj_KQT_mem(model,"Llama-2-7B_C4_train_128_2048",128,"Llama-2-7B_projKQT_C4_train_128_2048",save=True)

In [ ]:
#loading spectrum
spec = read_spectrums(model,"Llama-2-7B_projSVD_C4_train_128_2048")

In [ ]:
#computing ranks
eps = 0.1
ranks = computing_rank(model,spec,[[eps,eps] for _ in range(L)],square=True)
print(ranks)

In [ ]:
#loading projections
list_proj_eigen = read_proj(model,"Llama-2-7B_projSVDEigen_C4_train_128_2048")
list_proj_svd = read_proj(model,"Llama-2-7B_projSVD_C4_train_128_2048")
list_proj_kqt = read_proj(model,"Llama-2-7B_projKQT_C4_train_128_2048")

In [ ]:
#computing validation errors
s=32
error_svd = validation_error(model,valdata[:s],list_proj_svd,ranks,rope=True)
error_eigen = validation_error(model,valdata[:s],list_proj_eigen,ranks,rope=True)
error_kqt = validation_error(model,valdata[:s],list_proj_kqt,ranks,rope=True)

In [ ]:
def save_error(error, filename):
    torch.save(torch.tensor(error),filename)

def load_error(filename):
    return torch.load(filename).tolist()

In [ ]:
#experiment with unbalance factor
s=32
for alpha in [2,3,4,5,6,7,8,9,10]:
    list_proj_eigen = read_proj(model,"Llama-2-7B_projSVDEigen_C4_train_128_2048_"+str(alpha))
    print("Llama-2-7B_projSVDEigen_C4_train_128_2048_"+str(alpha))
    error_eigen = validation_error(model,valdata[:s],list_proj_eigen,ranks,rope=True)
    print(error_eigen[6])
    save_error(error_eigen,"error_llama2_7B_SVDEigen_valdata32_2048_C4_"+str(alpha)+".pt")

In [ ]:
#saving errors
save_error(error_kqt,"error_llama2_7B_KQT_valdata32_2048_C4.pt")
save_error(error_svd,"error_llama2_7B_SVD_valdata32_2048_C4.pt")
save_error(error_eigen,"error_llama2_7B_SVDEigen_valdata32_2048_C4.pt")